In [19]:
# The Truth Filtering Engine
# kcELECTRA model

In [20]:
!pip install -q transformers datasets accelerate scikit-learn pandas

In [21]:
# 데이터 처리용
import pandas as pd
import numpy as np

# 학습/검증 데이터 분리
from sklearn.model_selection import train_test_split

# 평가 지표 계산
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# Hugging Face Dataset 변환용
from datasets import Dataset, DatasetDict

# KcELECTRA tokenizer, 분류 모델, Trainer 불러오기
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# PyTorch
import torch

In [22]:
# 사용할 KcELECTRA 모델
# beomi/KcELECTRA-base = Korean comments ELECTRA 모델
# 즉, 한국어 댓글/구어체/noisy text에 맞춰 사전학습된 ELECTRA 계열 모델
MODEL_NAME = "beomi/KcELECTRA-base"

# 라벨 개수
# 0 = 비광고
# 1 = 광고
NUM_LABELS = 2

# 모델 출력 라벨 이름 지정
id2label = {
    0: "비광고",
    1: "광고"
}

label2id = {
    "비광고": 0,
    "광고": 1
}

In [23]:
import pandas as pd

# CSV 파일 경로
file_path = "/content/APIReviewList_rows.csv"

# CSV 읽기 (인코딩 지정)
df = pd.read_csv(file_path, encoding='cp949')

# 데이터 확인
df.head()

,id,review_description,is_ad
0,1,맛탕도 유명하던데 맛탕도 먹어봐야징~ 광교에서 즉석떢볶이 땡길때 즉석떡볶이 맛집 <...,0
1,2,광교 즉석떡볶이 고양이부엌에서 맛있게 드셔보시길 추천드리면서 포스팅 마치겠습니다:)...,0
2,3,"방문해봤습니다 고양이부엌은 체인점으로 동탄, 강남, 정자 등등 여러군데 있어요 ! ...",0
3,4,동네에 있는 <b>고양이 부엌</b> 맨날 가보자했다가 못갔는데 이번 기회에 가보자...,0
4,5,안녕하세요 오늘은 광교에서 맛있게 먹고온 즉떡 맛집 <b>고양이부엌 광교점</b>을...,0


In [24]:
# 네 CSV 컬럼명에 맞게 설정
TEXT_COL = "review_description"
LABEL_COL = "is_ad"

# 필요한 컬럼만 사용
df = df[[TEXT_COL, LABEL_COL]].copy()

# 결측치 제거
df = df.dropna()

# label을 정수형으로 변환
df[LABEL_COL] = df[LABEL_COL].astype(int)

# 모델 학습용 컬럼명으로 변경
df = df.rename(columns={
    TEXT_COL: "text",
    LABEL_COL: "label"
})

# 라벨 분포 확인
print(df["label"].value_counts())

df.head()

label
1    153
0     92
Name: count, dtype: int64


,text,label
0,맛탕도 유명하던데 맛탕도 먹어봐야징~ 광교에서 즉석떢볶이 땡길때 즉석떡볶이 맛집 <...,0
1,광교 즉석떡볶이 고양이부엌에서 맛있게 드셔보시길 추천드리면서 포스팅 마치겠습니다:)...,0
2,"방문해봤습니다 고양이부엌은 체인점으로 동탄, 강남, 정자 등등 여러군데 있어요 ! ...",0
3,동네에 있는 <b>고양이 부엌</b> 맨날 가보자했다가 못갔는데 이번 기회에 가보자...,0
4,안녕하세요 오늘은 광교에서 맛있게 먹고온 즉떡 맛집 <b>고양이부엌 광교점</b>을...,0


In [25]:
# 전체 데이터에서 train 80%, temp 20%로 분리
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]   # 광고/내돈내산 비율 유지
)

# temp 20%를 validation 10%, test 10%로 다시 분리
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test:", len(test_df))

train: 196
valid: 24
test: 25


In [26]:
# pandas DataFrame을 Hugging Face Dataset 형식으로 변환
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 196
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 24
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25
    })
})

In [27]:
# KcELECTRA tokenizer 불러오기
# 문장을 모델이 이해할 수 있는 숫자 토큰으로 바꿔준다.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# description은 보통 짧으므로 256 정도로 설정
# 본문 전체를 넣을 거면 512까지 고려 가능
MAX_LENGTH = 256

In [28]:
def tokenize_function(batch):
    """
    text 컬럼의 문장을 KcELECTRA 입력 형태로 변환하는 함수

    input_ids:
    문장을 숫자 토큰으로 바꾼 결과

    attention_mask:
    실제 문장 부분과 padding 부분을 구분하는 값

    truncation=True:
    길이가 MAX_LENGTH보다 길면 자름

    padding="max_length":
    모든 문장의 길이를 MAX_LENGTH로 맞춤
    """

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [29]:
# train, validation, test 전체에 토큰화 적용
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Trainer는 정답 컬럼 이름을 labels로 인식하는 경우가 많으므로 변경
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# PyTorch tensor 형태로 변환
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

tokenized_dataset

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 196
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 24
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 25
    })
})

In [30]:
# KcELECTRA-base 위에 분류용 head를 붙인다.
# num_labels=2 이므로 광고/내돈내산 이진 분류 모델이 된다.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoin

In [31]:
def compute_metrics(eval_pred):
    """
    validation/test 평가에 사용할 함수
    """

    logits, labels = eval_pred

    # logits 중 가장 큰 값의 index를 예측 라벨로 사용
    predictions = np.argmax(logits, axis=-1)

    # precision, recall, f1 계산
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    # accuracy 계산
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [32]:
training_args = TrainingArguments(
    # 학습 결과 저장 폴더
    output_dir="./kcelectra_ad_classifier",

    # 학습 epoch 수
    num_train_epochs=3,

    # batch size
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # learning rate
    learning_rate=2e-5,

    # 과적합 방지
    weight_decay=0.01,

    # epoch마다 평가
    eval_strategy="epoch",

    # epoch마다 모델 저장
    save_strategy="epoch",

    # 가장 좋은 모델을 마지막에 불러오기
    load_best_model_at_end=True,

    # f1 기준으로 best model 선택
    metric_for_best_model="f1",

    # f1은 높을수록 좋음
    greater_is_better=True,

    # 로그 출력
    logging_steps=50,

    # wandb 사용 안 함
    report_to="none"
)

In [33]:
evaluation_strategy="epoch"

In [34]:
trainer = Trainer(
    # 학습할 모델
    model=model,

    # 학습 설정
    args=training_args,

    # train 데이터
    train_dataset=tokenized_dataset["train"],

    # validation 데이터
    eval_dataset=tokenized_dataset["validation"],

    # 평가 함수
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.660812,0.625000,0.625000,1.000000,0.769231
2,No log,0.648125,0.625000,0.625000,1.000000,0.769231
3,No log,0.643879,0.625000,0.625000,1.000000,0.769231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=39, training_loss=0.6557467044928135, metrics={'train_runtime': 50.2101, 'train_samples_per_second': 11.711, 'train_steps_per_second': 0.777, 'total_flos': 77354650275840.0, 'train_loss': 0.6557467044928135, 'epoch': 3.0})

In [36]:
# test 데이터로 최종 성능 평가
test_result = trainer.evaluate(tokenized_dataset["test"])

test_result

{'eval_loss': 0.6524442434310913,
 'eval_accuracy': 0.64,
 'eval_precision': 0.64,
 'eval_recall': 1.0,
 'eval_f1': 0.7804878048780488,
 'eval_runtime': 0.4428,
 'eval_samples_per_second': 56.458,
 'eval_steps_per_second': 4.517,
 'epoch': 3.0}

In [37]:
# test 데이터 예측
pred_output = trainer.predict(tokenized_dataset["test"])

# 예측 결과
logits = pred_output.predictions
pred_labels = np.argmax(logits, axis=-1)

# 실제 정답
true_labels = pred_output.label_ids

# classification report 출력
print(classification_report(
    true_labels,
    pred_labels,
    target_names=["비광고", "광고"],
    digits=4
))

              precision    recall  f1-score   support

         비광고     0.0000    0.0000    0.0000         9
          광고     0.6400    1.0000    0.7805        16

    accuracy                         0.6400        25
   macro avg     0.3200    0.5000    0.3902        25
weighted avg     0.4096    0.6400    0.4995        25



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:
print("train_df 클래스별 데이터 개수:")
print(train_df['label'].value_counts())

train_df 클래스별 데이터 개수:
label
1    122
0     74
Name: count, dtype: int64


In [39]:
print("test_df 클래스별 데이터 개수:")
print(test_df['label'].value_counts())

test_df 클래스별 데이터 개수:
label
1    16
0     9
Name: count, dtype: int64


In [40]:
print("test_df 클래스별 데이터 개수:")
print(valid_df['label'].value_counts())

test_df 클래스별 데이터 개수:
label
1    15
0     9
Name: count, dtype: int64


### 1. 클래스 가중치 계산

`sklearn.utils.class_weight.compute_class_weight`를 사용하여 `train_df`의 `label` 분포에 따라 클래스 가중치를 계산합니다. 이 가중치는 `torch.Tensor` 형태로 변환하여 손실 함수에 전달됩니다.

In [41]:
from sklearn.utils.class_weight import compute_class_weight

# 클래스 가중치 계산
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)

# PyTorch Tensor로 변환
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class Weights:", class_weights_tensor)

Class Weights: tensor([1.3243, 0.8033])


### 2. Custom Trainer 정의

`Trainer` 클래스를 상속받아 `compute_loss` 메서드를 오버라이드하여, `torch.nn.CrossEntropyLoss`에 계산된 `class_weights_tensor`를 `weight` 인수로 전달합니다. 이렇게 하면 불균형한 클래스에 대한 학습 편향을 줄일 수 있습니다.

In [42]:
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # 클래스 가중치를 적용한 CrossEntropyLoss 사용
        # 모델이 있는 장치(CPU/GPU)에 가중치 텐서를 배치
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

### 3. Custom Trainer 인스턴스 생성 및 모델 재학습

새롭게 정의한 `CustomTrainer`를 사용하여 모델을 재학습시키겠습니다. `trainer` 객체를 다시 만들고 `class_weights`를 전달한 후, `trainer.train()`을 호출합니다.

In [43]:
# CustomTrainer 인스턴스 생성
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor # 계산된 클래스 가중치 전달
)

# 모델 재학습
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.661360,0.833333,0.789474,1.000000,0.882353
2,No log,0.612593,0.833333,0.789474,1.000000,0.882353
3,No log,0.607286,0.791667,0.812500,0.866667,0.838710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

TrainOutput(global_step=39, training_loss=0.6349947024614383, metrics={'train_runtime': 63.6193, 'train_samples_per_second': 9.242, 'train_steps_per_second': 0.613, 'total_flos': 77354650275840.0, 'train_loss': 0.6349947024614383, 'epoch': 3.0})

### 4. 재학습된 모델 평가

재학습된 모델의 성능을 `test` 데이터셋으로 다시 평가하고, `classification_report`를 출력하여 `비광고` 클래스의 점수가 개선되었는지 확인합니다.

In [44]:
# test 데이터로 최종 성능 평가
test_result_re_trained = trainer.evaluate(tokenized_dataset["test"])
print("Re-trained Model Test Result:", test_result_re_trained)

# test 데이터 예측
pred_output_re_trained = trainer.predict(tokenized_dataset["test"])

# 예측 결과
logits_re_trained = pred_output_re_trained.predictions
pred_labels_re_trained = np.argmax(logits_re_trained, axis=-1)

# 실제 정답
true_labels_re_trained = pred_output_re_trained.label_ids

# classification report 출력
print("\nClassification Report for Re-trained Model:")
print(classification_report(
    true_labels_re_trained,
    pred_labels_re_trained,
    target_names=["비광고", "광고"],
    digits=4
))

Re-trained Model Test Result: {'eval_loss': 0.6636599898338318, 'eval_accuracy': 0.72, 'eval_precision': 0.7647058823529411, 'eval_recall': 0.8125, 'eval_f1': 0.7878787878787878, 'eval_runtime': 0.4288, 'eval_samples_per_second': 58.306, 'eval_steps_per_second': 4.664, 'epoch': 3.0}

Classification Report for Re-trained Model:
              precision    recall  f1-score   support

         비광고     0.6250    0.5556    0.5882         9
          광고     0.7647    0.8125    0.7879        16

    accuracy                         0.7200        25
   macro avg     0.6949    0.6840    0.6881        25
weighted avg     0.7144    0.7200    0.7160        25



In [45]:
def predict_ad(text):
    """
    새로운 블로그 description을 넣으면
    광고/비광고 예측 결과를 반환하는 함수
    """

    # 입력 문장을 tokenizer로 변환
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length", # padding="longest"
        max_length=MAX_LENGTH
    )

    # 모델이 있는 장치로 입력 이동
    # `value.to(model.device)`: PyTorch 텐서인 `value`를 `model.device`가 가리키는 장치 (예: CPU 또는 GPU)로 이동시키는 메서드입니다.
    # 이렇게 함으로써 입력 데이터와 모델이 동일한 장치에 위치하여 올바른 연산이 가능하게 합니다.
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # 예측 모드
    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

        # logits을 확률로 변환
        probs = torch.softmax(outputs.logits, dim=-1)[0]

        # 가장 높은 확률의 라벨 선택
        pred_id = int(torch.argmax(probs).item())

    return {
        "prediction": id2label[pred_id],
        "prob_비광고": float(probs[0]),
        "prob_광고": float(probs[1])
    }

In [46]:
model.eval()

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [47]:
model

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [48]:
sample_text = """
<b>바나나테이블 광교점</b>에 가서 여사장님께서 궁금하셨던 이유에 대해 알려드릴게요! ✈️ 여러분, 비행기... <b>바나나테이블 광교점</b>에서 현지 맛을 느껴보려는 여행자들에게 추천할만한 곳이에요. 가격은 비싸지만...
"""

predict_ad(sample_text)

{'prediction': '광고',
 'prob_비광고': 0.4966886639595032,
 'prob_광고': 0.5033113956451416}

In [49]:
# 학습 완료된 모델 저장
SAVE_DIR = "./electra_ad_classifier"

try:
    trainer.save_model(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print("저장 완료:", SAVE_DIR)
except NameError as e:
    print(f"Error: {e}. 'trainer' or 'tokenizer' is not defined.")
    print("Please ensure that the model training cells (especially cell 3714b048, where 'trainer' is initialized) and the tokenizer initialization cell (cell Se6uHVAyTu5j) have been executed.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

저장 완료: ./electra_ad_classifier


In [51]:
import shutil
import os

# 압축할 디렉토리 경로
source_dir = SAVE_DIR
# 압축 파일 이름 (확장자 제외)
output_filename = source_dir

try:
    shutil.make_archive(output_filename, 'zip', source_dir)
    print(f"'{source_dir}' 폴더가 '{output_filename}.zip'으로 압축되었습니다.")
    # 파일이 생성되었는지 확인
    if os.path.exists(f'{output_filename}.zip'):
        print(f"'{output_filename}.zip' 파일을 Colab 좌측 파일 탐색기에서 찾아 다운로드할 수 있습니다.")
    else:
        print("압축 파일 생성에 실패했습니다.")
except Exception as e:
    print(f"압축 중 오류 발생: {e}")

'./electra_ad_classifier' 폴더가 './electra_ad_classifier.zip'으로 압축되었습니다.
'./electra_ad_classifier.zip' 파일을 Colab 좌측 파일 탐색기에서 찾아 다운로드할 수 있습니다.


In [50]:
print(model.device)

cuda:0
